# NYC 311 Service Requests — Dataset Exploration

## ICS 3202 Semester Project · Deliverable 1
**Aug–Nov 2026**

---

**Members:**

| Name | Admission number |
|------|------------------|
| Atogo Carl | 191245 |
| Bill Kevin | 189725 |
| Segera Rooney | 189735 |

---

### Project

A mobile application that connects tenants of a residential estate to household
service providers supported by a machine learning engine.

### What the ML engine must do

A tenant types a plain-language description of a problem, for example
*"kitchen tap has been dripping since Monday"*. The engine must then decide:

1. **which service category** 
2. **roughly how long it should take to resolve**



---
# Part 1 — Dataset discovery

We searched open-data portals, Kaggle, and municipal data repositories for
datasets in which a resident reports a household or building problem that is
then categorised and resolved — the same pipeline our application implements.
Three datasets were identified as relevant to our research area.

## Dataset 1 — NYC 311 Service Requests (2020 to Present)

**Source:** NYC Open Data
**Link:** https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9
**Scale:** Over 24 million records
**Access:** Open Socrata API and CSV export; no account required
**Licence:** Public domain

New York City's 311 line receives non-emergency service requests from residents,
covering household and building problems such as heating and hot water failures,
plumbing faults, electrical issues, appliance breakdowns and pest complaints.
Each record holds the complaint type, a short descriptor of the problem, the
responding agency, the location, and timestamps for when the request was created
and closed.

**Relevance:** This is the closest large-scale analogue to our application. A
resident reports a household problem in their own words, the request is
categorised, routed to a responder, and closed after some duration — exactly the
flow a tenant follows in our app.

## Dataset 2 — Housing Maintenance Requests (FCC)

**Source:** Fingal County Council, Ireland (data.gov.ie / Smart Dublin)
**Link:** https://data.gov.ie/dataset/housing-maintenance-requests-fcc1
**Coverage:** 2018–2024 inclusive
**Access:** ArcGIS Hub dataset
**Licence:** Creative Commons Attribution 4.0

This dataset records repair requests received by the **Estate Management Team**
of Fingal County Council's Housing Division, relating to the council's housing
stock. Requests are categorised by type, including Heating and Plumbing.

**Relevance:** Thematically this is the closest match found anywhere in open
data — it is literally an estate management team logging household repair
requests from tenants, which is precisely our application's domain.

**Limitation:** The dataset appears to be published as aggregate counts rather
than individual request records. Without record-level data there are no
individual examples to train a model on, which restricts its use to contextual
evidence rather than training.

## Dataset 3 — Chicago 311 Service Requests

**Source:** City of Chicago Data Portal
**Link:** https://data.cityofchicago.org/Service-Requests/311-Service-Requests/v6vf-nfxy
**Scale:** Approximately 5 million records
**Access:** Open Socrata API and CSV export
**Licence:** Public domain

Chicago's equivalent 311 system, recording non-emergency service requests from
residents. Each record contains the service request type, its current status,
the responsible department, location details, and creation and completion dates.

**Relevance:** Structurally similar to the NYC dataset and drawn from a different
city, making it useful both as an alternative training source and as a way to
test whether a model trained on one municipality transfers to another — a
question directly related to our research gap.



# Part 2 — Expected output variable for each dataset

The **output variable** (or target) is the column the model must learn to
predict. For each of the three datasets identified above, we name at least one
variable that would serve as the expected output of our final application.

## Dataset 1 — NYC 311 Service Requests

| Variable | Type | Role as the application's output |
|---|---|---|
| **`complaint_type`** | Categorical | The service category predicted from the tenant's free-text description, so the request routes to the correct provider without the tenant selecting a category manually |
| `resolution_hours` (derived from `closed_date − created_date`) | Continuous | The predicted time to resolve, used to set the tenant's expectations and to prioritise the estate's queue |
| `agency` | Categorical | The responding body — analogous to predicting which type of provider should be assigned |

## Dataset 2 — Housing Maintenance Requests (FCC)

| Variable | Type | Role as the application's output |
|---|---|---|
| **Request category** (Heating, Plumbing, etc.) | Categorical | The maintenance category predicted from a tenant's reported problem |

## Dataset 3 — Chicago 311 Service Requests

| Variable | Type | Role as the application's output |
|---|---|---|
| **`sr_type`** | Categorical | The service request type predicted from the tenant's description |
| Days to completion (derived from creation and completion dates) | Continuous | The predicted resolution time |
| `owner_department` | Categorical | The department responsible — analogous to provider assignment |

---

## Dataset selected: NYC 311 Service Requests

Comparing the three against what our engine needs:

| Criterion | NYC 311 | Fingal FCC | Chicago 311 |
|---|---|---|---|
| Record-level data (individual requests) | Yes | No — appears aggregated | Yes |
| Volume | 24M+ | Small | ~5M |
| Categorical target present | Yes | Yes | Yes |
| Continuous target derivable | Yes | No | Yes |
| Free-text problem description | Yes (`descriptor`) | No | Limited |
| Open API, no login | Yes | ArcGIS Hub only | Yes |

**Fingal FCC** is thematically the strongest match — it is estate maintenance
data from an estate management team — but without record-level rows there are no
individual examples for a model to learn from, so it cannot serve as our
training set.

**Chicago 311** is a viable alternative and we retain it as a secondary source,
particularly for testing whether a model trained on one city transfers to
another.

**NYC 311 is selected** as our primary dataset. It is the largest of the three,
provides both a categorical and a continuous target, includes a free-text
descriptor column resembling what a tenant would actually type, and is
accessible through an open API requiring no login — which keeps this notebook
fully reproducible.


In [1]:
import pandas as pd

# Fetch a working sample via the NYC Open Data API
url = "https://data.cityofnewyork.us/resource/erm2-nwe9.csv?$limit=10000"
df = pd.read_csv(url)

rows, cols = df.shape
print(f"The dataset sample contains {rows} rows and {cols} columns.")

The dataset sample contains 10000 rows and 44 columns.


In [2]:
# Show pandas datatypes for all columns
print("Datatypes contained in the dataset:")
print(df.dtypes)

Datatypes contained in the dataset:
unique_key                          int64
created_date                          str
closed_date                           str
agency                                str
agency_name                           str
complaint_type                        str
descriptor                            str
descriptor_2                          str
location_type                         str
incident_zip                      float64
incident_address                      str
street_name                           str
cross_street_1                        str
cross_street_2                        str
intersection_street_1                 str
intersection_street_2                 str
address_type                          str
city                                  str
landmark                              str
facility_type                         str
status                                str
due_date                              str
resolution_description                st

In [3]:
# Check for missing/null values in the dataframe
is_complete = not df.isnull().values.any()

# Total missing values for context
total_missing = df.isnull().sum().sum()

print(f"Is the dataset complete? {is_complete}")
print(f"Total missing values: {total_missing}")

Is the dataset complete? False
Total missing values: 126886


In [4]:
first_15 = df.head(15)
last_20 = df.tail(20)

# Concatenate the two slices along the row axis
df_sample = pd.concat([first_15, last_20])

df_sample

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,...,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,70346824,2026-09-09T02:28:03.000,NaN,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,11357.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,70341834,2026-09-09T01:51:11.000,NaN,NYPD,New York City Police Department,Illegal Parking,Blocked Crosswalk,NaN,Street/Sidewalk,11377.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.735713,-73.898572,POINT (-73.898571883689 40.735713173606)
2,70341778,2026-09-09T01:51:00.000,NaN,NYPD,New York City Police Department,Illegal Parking,Blocked Hydrant,NaN,Street/Sidewalk,11379.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.728835,-73.873377,POINT (-73.873376756444 40.728834567303)
3,70343491,2026-09-09T01:50:41.000,NaN,NYPD,New York City Police Department,Blocked Driveway,No Access,NaN,Street/Sidewalk,11433.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.687317,-73.793451,POINT (-73.793450712899 40.68731680994)
4,70343406,2026-09-09T01:50:29.000,NaN,NYPD,New York City Police Department,Blocked Driveway,No Access,NaN,Street/Sidewalk,11385.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.704653,-73.904632,POINT (-73.90463152072 40.70465315031)
5,70347918,2026-09-09T01:50:09.000,NaN,DPR,Department of Parks and Recreation,Illegal Tree Damage,Trunk Damaged,NaN,Street,10024.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.784880,-73.973358,POINT (-73.973357675909 40.784880179959)
6,70344154,2026-09-09T01:49:38.000,NaN,NYPD,New York City Police Department,Illegal Parking,Blocked Hydrant,NaN,Street/Sidewalk,11204.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.618696,-73.993044,POINT (-73.993044423749 40.618695778532)
7,70342778,2026-09-09T01:49:34.000,NaN,NYPD,New York City Police Department,Illegal Fireworks,NaN,NaN,Street/Sidewalk,11238.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.689100,-73.958024,POINT (-73.958024153332 40.689100213338)
8,70349644,2026-09-09T01:49:24.000,NaN,NYPD,New York City Police Department,Illegal Parking,Commercial Overnight Parking,NaN,Street/Sidewalk,11423.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.721389,-73.765307,POINT (-73.765306688242 40.721388635013)
9,70341016,2026-09-09T01:49:19.000,NaN,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11212.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.662317,-73.921442,POINT (-73.921441772013 40.662316746945)
